# Notebook 4: Training Classifiers & Experiment Tracking

This notebook explains **what a classifier is, how Logistic Regression and XGBoost work, how we evaluate them, and why we track everything in MLflow.**

Covers the design decisions in `train.py`.

---

## What Are We Trying to Do?

At this point in the pipeline:
- BERTopic has **discovered** 19 topics and labeled 87k articles
- Each article has a TF-IDF feature vector (50,000 numbers)
- Each article has a topic label (0–18)

**Goal:** Train a model that can look at a *new* article's TF-IDF vector and predict which topic it belongs to — without needing to run BERTopic again.

This is called **supervised classification** — we're learning a mapping from features → labels using labeled examples.

```
Training:
  [TF-IDF vector] + [known topic label]  →  model learns the pattern

Inference (new article):
  [TF-IDF vector]  →  model predicts topic
```

**Why not just use BERTopic every time?**  
BERTopic takes minutes per batch. A trained Logistic Regression classifies in milliseconds. The hybrid approach uses BERTopic once to generate labels, then trains a fast classifier for production use.

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style='whitegrid', palette='muted')

# Load the data
df_clean   = pd.read_parquet('../data/processed/articles_clean.parquet', columns=['id'])
df_labeled = pd.read_parquet('../data/processed/articles_with_topics.parquet', columns=['id','topic_id','topic_label'])
tfidf_full = sp.load_npz('../data/processed/tfidf_matrix.npz')

# Align TF-IDF rows with labeled articles
df_clean['row_idx'] = np.arange(len(df_clean))
df_merged = df_labeled.merge(df_clean, on='id', how='inner')

X = tfidf_full[df_merged['row_idx'].values, :]
le = LabelEncoder()
y = le.fit_transform(df_merged['topic_id'].values)

print(f'Feature matrix X: {X.shape}  ({X.shape[0]:,} articles × {X.shape[1]:,} TF-IDF features)')
print(f'Labels y:         {len(y):,} samples, {len(le.classes_)} classes')

---

## The Train/Test Split — Why?

We hold back 20% of the data before training. The model **never sees this data during training** — it's used only to measure how well the model generalizes to new, unseen articles.

**Why not evaluate on training data?**  
A model can memorize the training data and score 100% — but then fail on new articles. This is called **overfitting**. The test set catches overfitting.

Think of it like studying for an exam with practice problems (train), then taking the real exam (test) — the real exam has questions you've never seen.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]:,} articles  (80%)')
print(f'Test:  {X_test.shape[0]:,} articles  (20%)')

# Show that stratify= preserved class proportions
train_dist = pd.Series(y_train).value_counts(normalize=True).sort_index()
test_dist  = pd.Series(y_test).value_counts(normalize=True).sort_index()

print(f'\nClass proportion check (should be nearly identical):')
comparison = pd.DataFrame({'train_%': (train_dist*100).round(1), 'test_%': (test_dist*100).round(1)})
print(comparison.head(8).to_string())

### Why `stratify=y`?

Some topics have only ~150 articles. Without stratification, a random split might put most of a small class in train and none in test — making that class's evaluation meaningless.

Stratification ensures every class has ~80% of its articles in train and ~20% in test, regardless of class size.

---

## Model 1: Logistic Regression (Baseline)

### What is Logistic Regression?

Despite the name, **Logistic Regression is a classifier**, not a regression model. The "regression" refers to the internal math — it regresses on the log-odds of each class.

**Intuition:**  
For each class (topic), LR learns a **weight** for every feature (word). At prediction time, it multiplies each TF-IDF score by its learned weight and sums them up. The class with the highest sum wins.

```
score(topic=Economics) = w₁×tfidf("federal") + w₂×tfidf("reserve") + w₃×tfidf("rate") + ...
score(topic=Sports)    = w₁×tfidf("nfl") + w₂×tfidf("touchdown") + w₃×tfidf("game") + ...

predicted_topic = argmax(all scores)
```

In [ ]:
# ── Build intuition with a tiny 2-class toy example ────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer

toy_docs = [
    # Economics articles
    "federal reserve raised interest rates quarter point",
    "stock market fell inflation concerns bond yield",
    "gdp growth slowed recession fears unemployment rate",
    "central bank policy monetary stimulus economic",
    # Sports articles
    "nfl touchdown game quarterback pass football",
    "basketball nba finals lakers score court game",
    "baseball world series pitcher strike home run",
    "soccer world cup goal match team stadium",
]
toy_labels = [0, 0, 0, 0, 1, 1, 1, 1]  # 0=economics, 1=sports

toy_vec = TfidfVectorizer()
X_toy = toy_vec.fit_transform(toy_docs)

toy_lr = LogisticRegression()
toy_lr.fit(X_toy, toy_labels)

# Show the learned weights for each word
feature_names = toy_vec.get_feature_names_out()
# Positive coefficient = word pushes toward class 1 (sports)
# Negative coefficient = word pushes toward class 0 (economics)
coef_df = pd.DataFrame({'word': feature_names, 'weight': toy_lr.coef_[0]}).sort_values('weight')

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if w < 0 else 'coral' for w in coef_df['weight']]
ax.barh(coef_df['word'], coef_df['weight'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression Learned Weights (Toy Example)', fontsize=13)
ax.set_xlabel('Weight  (negative=Economics, positive=Sports)')
plt.tight_layout()
plt.show()

In [ ]:
# Now predict on a new article
new_article = ["the quarterback threw a touchdown in the final quarter"]
new_vec = toy_vec.transform(new_article)
pred = toy_lr.predict(new_vec)[0]
prob = toy_lr.predict_proba(new_vec)[0]

print(f'New article: "{new_article[0]}"')
print(f'Predicted:   {"Sports" if pred == 1 else "Economics"}')
print(f'Confidence:  Economics={prob[0]:.1%}  Sports={prob[1]:.1%}')

### The `C` Regularization Parameter

**Regularization** prevents overfitting by penalizing models with very large weights. Imagine a model that learned: "if 'quarterback' appears → definitely Sports with 99.99% confidence". That's overconfident and brittle.

- **Small C** (e.g. 0.01) → strong penalty → small weights → underfitting (too cautious)
- **Large C** (e.g. 100) → weak penalty → large weights → overfitting (overconfident)
- **C=5** (our choice) → balanced — works well for TF-IDF text features

In [ ]:
# Demonstrate effect of C on a small subset of real data
X_mini = X[:5000]
y_mini = y[:5000]
Xm_train, Xm_test, ym_train, ym_test = train_test_split(X_mini, y_mini, test_size=0.2, random_state=42, stratify=y_mini)

C_values = [0.01, 0.1, 1, 5, 10, 50]
results = []
for C in C_values:
    m = LogisticRegression(C=C, max_iter=500, solver='lbfgs', multi_class='multinomial', n_jobs=-1)
    m.fit(Xm_train, ym_train)
    train_acc = accuracy_score(ym_train, m.predict(Xm_train))
    test_acc  = accuracy_score(ym_test, m.predict(Xm_test))
    results.append({'C': C, 'train_acc': train_acc, 'test_acc': test_acc})

res_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(res_df['C'], res_df['train_acc'], 'o-', label='Train accuracy', color='steelblue')
ax.semilogx(res_df['C'], res_df['test_acc'],  's-', label='Test accuracy',  color='coral')
ax.axvline(5, color='green', linestyle='--', label='Our choice (C=5)')
ax.set_xlabel('C (regularization strength)')
ax.set_ylabel('Accuracy')
ax.set_title('Effect of Regularization (C) on Accuracy', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

> **Interview Talking Point:**  
> *"I used Logistic Regression as the baseline because it's the standard for text classification — it handles sparse high-dimensional TF-IDF features well, trains in seconds, and is interpretable. The `C` parameter controls regularization: too small and the model underfits, too large and it overfits. C=5 is a well-established default for TF-IDF text."*

---

## Model 2: XGBoost (Challenger)

### What is XGBoost?

XGBoost builds an **ensemble of decision trees** sequentially. Each tree learns to correct the mistakes of all the trees before it. This is called **gradient boosting**.

In [ ]:
# ── Decision tree intuition ──────────────────────────────────────────────────
# A single decision tree is like a flowchart of yes/no questions

print('Toy decision tree (conceptual):')
print()
print('  Does the article contain "nfl" or "touchdown"?')
print('    YES → Sports')
print('    NO  → Does it contain "federal" or "reserve"?')
print('            YES → Economics')
print('            NO  → Does it contain "trump" or "senate"?')
print('                    YES → Politics')
print('                    NO  → ... (keep splitting)')
print()
print('XGBoost builds 100+ such trees, each fixing the errors of the previous ones.')
print('Final prediction = weighted vote across all trees.')

In [ ]:
# ── Gradient boosting intuition ──────────────────────────────────────────────
# Each new tree focuses on the articles the previous trees got WRONG

print('Gradient boosting in 3 steps:')
print()
print('  Tree 1: Trained on all data.')
print('          Gets most articles right, but struggles with ambiguous ones.')
print()
print('  Tree 2: Trained with higher weight on articles Tree 1 got WRONG.')
print('          Focuses on hard cases.')
print()
print('  Tree 3: Same — focuses on what Trees 1+2 still get wrong.')
print()
print('  ... repeat for 100 trees ...')
print()
print('  Final prediction: weighted combination of all 100 trees.')
print()
print('Learning rate=0.1 means each tree contributes only 10% of its prediction,')
print('preventing any single tree from dominating (reduces overfitting).')

### Why might LR outperform XGBoost here?

XGBoost wins on many tabular datasets, but **TF-IDF text features are linear in nature** — the relationship between word presence and topic is approximately linear. LR is purpose-built for linear problems.

Decision trees also struggle with very high-dimensional sparse features: most of 50,000 feature values are zero, and trees have to find splits across all 50,000 dimensions. LR handles this natively.

> **Interview Talking Point:**  
> *"I compared Logistic Regression against XGBoost. For TF-IDF text features, LR is often competitive or better than tree-based methods because the word→topic relationship is approximately linear. XGBoost shines on dense tabular data with non-linear interactions. The comparison in MLflow shows exactly which model performed better on this dataset."*

---

## Evaluation Metrics — Accuracy vs F1

### Why not just use accuracy?

In [ ]:
# Our class distribution is imbalanced — some topics have 10x more articles than others
topic_counts = df_merged['topic_label'].str.split('_').str[1:].str.join(' ').value_counts()

fig, ax = plt.subplots(figsize=(11, 5))
topic_counts.sort_values().plot(kind='barh', ax=ax, color=sns.color_palette('muted', len(topic_counts)))
ax.set_title('Class Imbalance: Articles per Topic', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Articles')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

max_count = topic_counts.max()
min_count = topic_counts.min()
print(f'Largest topic:  {max_count:,} articles')
print(f'Smallest topic: {min_count:,} articles')
print(f'Imbalance ratio: {max_count/min_count:.1f}x')

In [ ]:
# Why imbalance matters for accuracy
print('Imbalanced class problem (toy example):')
print()
print('  Suppose topic A has 10,000 articles, topic B has 100 articles.')
print('  A model that ALWAYS predicts topic A achieves 99% accuracy.')
print('  But it gets topic B wrong 100% of the time!')
print()
print('  Accuracy = (correct / total) — misleading with imbalanced classes')
print()
print('  F1 (macro) = average F1 score across ALL classes, equally weighted')
print('  This penalizes the model for ignoring small classes.')

In [ ]:
# What is F1?
print('F1 score = harmonic mean of Precision and Recall')
print()
print('  Precision = of all articles I predicted as "Sports", what % were actually Sports?')
print('              (how often am I right when I say Sports?)')
print()
print('  Recall    = of all actual Sports articles, what % did I correctly identify?')
print('              (how many Sports articles did I catch?)')
print()
print('  F1 = 2 × (Precision × Recall) / (Precision + Recall)')
print('       punishes models that sacrifice one for the other')
print()
print('  Macro F1 = average F1 across all topics (each topic weighted equally)')
print('  This is our primary metric because topics have different sizes.')

---

## MLflow — Experiment Tracking

### What problem does MLflow solve?

During ML development you run many experiments: different models, different hyperparameters, different features. Without tracking, you lose track of what you tried and what worked.

MLflow automatically logs:
- **Parameters** — what settings did I use?
- **Metrics** — what accuracy/F1 did I get?
- **Artifacts** — the trained model file
- **Source code** — which version of the code produced this run?

Think of it as **version control for ML experiments**.

In [ ]:
import mlflow

# After running train.py, explore the logged runs
mlflow.set_tracking_uri('../mlruns')  # tell mlflow where the run logs are

try:
    experiment = mlflow.get_experiment_by_name('news-topic-classification')
    if experiment:
        runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
        if len(runs) > 0:
            cols = ['tags.mlflow.runName', 'params.model', 'metrics.accuracy', 'metrics.f1_macro', 'start_time']
            available = [c for c in cols if c in runs.columns]
            print('Logged MLflow runs:')
            print(runs[available].to_string(index=False))
        else:
            print('No runs yet — run train.py first, then re-run this cell.')
    else:
        print('Experiment not found — run train.py first.')
except Exception as e:
    print(f'Run train.py first to populate MLflow runs. ({e})')

In [ ]:
# After running train.py, you can launch the MLflow UI to see a visual dashboard
print('To view the MLflow dashboard:')
print()
print('  1. Open a new terminal')
print('  2. cd to your project folder')
print('  3. Run: source .venv/bin/activate && mlflow ui')
print('  4. Open: http://127.0.0.1:5000')
print()
print('The UI shows:')
print('  • Both model runs side-by-side')
print('  • All logged parameters and metrics')
print('  • Downloadable model artifacts')
print('  • Chart comparison across runs')

> **Interview Talking Point (MLflow):**  
> *"I tracked every training run in MLflow — parameters, metrics, and model artifacts. This lets me objectively compare Logistic Regression vs XGBoost on the same test set, reproduce any run exactly, and show a clear audit trail of my experimentation. In a team setting, MLflow means no one has to manually track 'which model did we deploy last Tuesday and what were its metrics?'"*

---

## After Running train.py — Interpreting Results

Run this cell after `python -m src.train` completes:

In [ ]:
# Load the trained LR model and show its per-class performance
import pickle
from pathlib import Path

model_path = Path('../data/processed/models/logistic_regression.pkl')
le_path    = Path('../data/processed/models/label_encoder.pkl')

if model_path.exists():
    lr_model = pickle.load(open(model_path, 'rb'))
    le_model = pickle.load(open(le_path, 'rb'))

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    y_pred = lr_model.predict(X_test)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(13, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=le_model.classes_, yticklabels=le_model.classes_)
    ax.set_title('Confusion Matrix — Logistic Regression', fontsize=14)
    ax.set_ylabel('True Topic')
    ax.set_xlabel('Predicted Topic')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    print('\nPer-class classification report:')
    print(classification_report(y_test, y_pred, target_names=[str(c) for c in le_model.classes_]))
else:
    print('Model not found — run train.py first, then re-run this cell.')

### How to read the confusion matrix

- **Diagonal** = correct predictions (true topic = predicted topic)
- **Off-diagonal** = mistakes (row = true topic, column = predicted topic)
- A bright off-diagonal cell means the model frequently confuses two specific topics — usually because they share vocabulary (e.g. US politics and Clinton/Trump topics)

---

## Summary

```
87,392 labeled articles
       │
       ├── 80% train  (69,913 articles)
       └── 20% test   (17,479 articles)  ← model never sees this during training
              │
     ┌────────┴────────┐
     │                 │
 Logistic         XGBoost
 Regression       (gradient-boosted trees)
 (linear model)   (non-linear model)
     │                 │
     └────────┬────────┘
              │
         MLflow logs both runs
         → compare accuracy + F1 macro
         → best model saved to disk
```

| Concept | One-line explanation |
|---------|---------------------|
| Train/test split | Hold back 20% so you can measure generalization |
| Stratify | Preserve class proportions in both splits |
| LR | Learns a weight per word per class; fast on sparse text |
| C parameter | Controls regularization strength — prevents overfitting |
| XGBoost | Ensemble of trees, each correcting the last |
| Accuracy | % correct — misleading with imbalanced classes |
| F1 macro | Average F1 per class — penalizes ignoring small topics |
| MLflow | Version control for ML experiments |